In [ ]:
!pip install anthropic pillow pandas -q

import json
import os
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets
from google.colab import drive

drive.mount('/content/drive')

IMAGE_FOLDER = '/content/drive/MyDrive/CDT_Dataset/images'
LABEL_PATH   = '/content/drive/MyDrive/CDT_Dataset/labels/ccss_labels_v2.json'
FINAL_PATH   = '/content/drive/MyDrive/CDT_Dataset/labels/ccss_labels_final.json'

with open(LABEL_PATH, 'r') as f:
    labels = json.load(f)

if os.path.exists(FINAL_PATH):
    with open(FINAL_PATH, 'r') as f:
        final_labels = json.load(f)
    print(f'โหลด progress human check: {len(final_labels)} รูป')
else:
    final_labels = {}
    print('human check')

print(f'โหลด labels ทั้งหมด: {len(labels)} รูป')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
โหลด progress human check: 1639 รูป
โหลด labels ทั้งหมด: 6602 รูป


In [ ]:
def data_summary(filename, data):
    return {
        'total': data.get('total_score', 0),
        'confidence': data.get('confidence', 1.0),
        'flags': data.get('flags', []),
        'scores': data.get('scores', {}),
        'reasons': []
    }

In [ ]:
import random

review_list = []

# กลุ่ม 1 — low score review ทุกรูป
low_score = [
    {'filename': f, **data_summary(f, d)}
    for f, d in labels.items()
    if d.get('total_score', 0) <= 1
    and f not in final_labels
]

# กลุ่ม 2 — low confidence review ทุกรูป
low_conf = [
    {'filename': f, **data_summary(f, d)}
    for f, d in labels.items()
    if d.get('confidence', 1) < 0.7
    and d.get('total_score', 0) > 1
    and f not in final_labels
]

# กลุ่ม 3 — high score สุ่มแค่ 200 รูป
high_score_all = [
    {'filename': f, **data_summary(f, d)}
    for f, d in labels.items()
    if d.get('total_score', 0) >= 9
    and f not in final_labels
]
high_score_sample = random.sample(
    high_score_all,
    min(200, len(high_score_all))
)

# กลุ่ม 4 — rotated review ทุกรูป
rotated = [
    {'filename': f, **data_summary(f, d)}
    for f, d in labels.items()
    if d.get('rotation_used', 0) != 0
    and f not in final_labels
]

review_list = low_score + low_conf + high_score_sample + rotated

seen = set()
review_list = [
    x for x in review_list
    if x['filename'] not in seen
    and not seen.add(x['filename'])
]

print(f'  low score (≤1)     : {len(low_score)} รูป')
print(f'  low confidence      : {len(low_conf)} รูป')
print(f'  high score (สุ่ม)  : {len(high_score_sample)} รูป → สุ่มจาก {len(high_score_all)}')
print(f'  rotated             : {len(rotated)} รูป')
print(f'\nรวม review: {len(review_list)} รูป')
print(f'review แล้ว: {len(final_labels)} รูป')

แบ่งตามกลุ่ม:
  low score (≤1)     : 0 รูป
  low confidence      : 0 รูป
  high score (สุ่ม)  : 200 รูป → สุ่มจาก 4387
  rotated             : 34 รูป

รวม review: 233 รูป
review แล้ว: 1406 รูป


In [ ]:
DOMAINS = ['digit_count', 'worst_quadrant_error',
           'spatial_arrangement', 'hands_present', 'hands_placement']
DOMAIN_LABELS = ['A: เลข 1-12', 'B: quadrant แย่สุด',
                 'C: ตำแหน่ง+ลำดับ', 'D: เข็ม', 'E: ตำแหน่งเข็ม']

state = {'index': 0}

out_image  = widgets.Output()
out_info   = widgets.Output()
out_status = widgets.Output()

progress_bar = widgets.IntProgress(
    value=len(final_labels),
    min=0, max=len(review_list) + len(final_labels),
    description='Progress:',
    style={'bar_color': '#4CAF50'},
    layout=widgets.Layout(width='500px')
)
progress_label = widgets.Label(
    value=f'Review แล้ว {len(final_labels)}/{len(review_list)+len(final_labels)} รูป'
)

sliders = {}
for key, label in zip(DOMAINS, DOMAIN_LABELS):
    sliders[key] = widgets.IntSlider(
        value=2, min=0, max=2, step=1,
        description=label,
        style={'description_width': '160px'},
        layout=widgets.Layout(width='450px')
    )

total_display = widgets.HTML(value='')

def update_total(change=None):
    total = sum(s.value for s in sliders.values())
    color = '#4CAF50' if total >= 6 else '#F44336'
    label = 'ปกติ' if total >= 6 else '⚠️ เสี่ยง Dementia'
    total_display.value = (
        f'<div style="font-size:16px;font-weight:bold;'
        f'color:{color};padding:8px;background:#f5f5f5;'
        f'border-radius:6px">คะแนนรวม: {total}/10 — {label}</div>'
    )

for s in sliders.values():
    s.observe(update_total, names='value')

btn_confirm = widgets.Button(
    description='ยืนยัน',
    button_style='success',
    layout=widgets.Layout(width='160px')
)
btn_skip = widgets.Button(
    description='ข้าม',
    button_style='info',
    layout=widgets.Layout(width='160px')
)
btn_back = widgets.Button(
    description='ย้อนกลับ',
    button_style='warning',
    layout=widgets.Layout(width='130px')
)

def show_current():
    if state['index'] >= len(review_list):
        with out_status:
            clear_output()
            print('Review all photo!')
        with out_image:
            clear_output()
        with out_info:
            clear_output()
        return

    item = review_list[state['index']]
    filename = item['filename']
    llm_scores = item['scores']

    with out_image:
        clear_output(wait=True)
        img = Image.open(f'{IMAGE_FOLDER}/{filename}')
        img.thumbnail((480, 480))
        img.save('/tmp/review.jpg', 'JPEG')
        display(widgets.Image(
            value=open('/tmp/review.jpg','rb').read(),
            format='jpeg',
            layout=widgets.Layout(width='480px')
        ))

    with out_info:
        clear_output(wait=True)
        print(f'รูปที่ {state["index"]+1}/{len(review_list)}')
        print(f'ไฟล์: {filename}')
        print(f'LLM ให้: {item["total"]}/10  confidence={item["confidence"]:.2f}')
        print(f'สาเหตุที่ review: {", ".join(item["reasons"])}')
        print(f'\nคะแนน LLM:')
        for domain, label in zip(DOMAINS, DOMAIN_LABELS):
            score = llm_scores.get(domain, '?')
            print(f'  {label}: {score}')

    for domain in DOMAINS:
        sliders[domain].value = llm_scores.get(domain, 2)

    update_total()

    with out_status:
        clear_output()

def save_and_next(btn=None):
    if state['index'] >= len(review_list):
        return
    item = review_list[state['index']]
    filename = item['filename']

    # บันทึกคะแนน
    new_scores = {d: sliders[d].value for d in DOMAINS}
    total = sum(new_scores.values())

    final_labels[filename] = {
        'image_id': filename,
        'scores': new_scores,
        'total_score': total,
        'risk_flag': total < 6,
        'confidence': item['confidence'],
        'flags': item['reasons'],
        'reviewer': 'human'
    }

    with open(FINAL_PATH, 'w') as f:
        json.dump(final_labels, f, ensure_ascii=False, indent=2)

    progress_bar.value = len(final_labels)
    progress_label.value = (f'Review แล้ว {len(final_labels)}/'
                            f'{len(review_list)+len(final_labels)} รูป')

    state['index'] += 1
    show_current()

def skip_and_next(btn=None):
    if state['index'] >= len(review_list):
        return
    item = review_list[state['index']]
    filename = item['filename']

    final_labels[filename] = {
        **labels[filename],
        'reviewer': 'human_approved'
    }

    with open(FINAL_PATH, 'w') as f:
        json.dump(final_labels, f, ensure_ascii=False, indent=2)

    progress_bar.value = len(final_labels)
    progress_label.value = (f'Review แล้ว {len(final_labels)}/'
                            f'{len(review_list)+len(final_labels)} รูป')

    state['index'] += 1
    show_current()

def go_back(btn=None):
    if state['index'] > 0:
        state['index'] -= 1
        show_current()

btn_confirm.on_click(save_and_next)
btn_skip.on_click(skip_and_next)
btn_back.on_click(go_back)

right_panel = widgets.VBox([
    out_info,
    widgets.HTML('<hr>'),
    widgets.HTML('<b>ปรับคะแนน:</b>'),
    *sliders.values(),
    widgets.HTML('<hr>'),
    total_display,
    widgets.HTML('<hr>'),
    widgets.HBox([btn_back, btn_skip, btn_confirm])
], layout=widgets.Layout(padding='12px', width='500px'))

header = widgets.VBox([
    widgets.HTML('<h2>Review</h2>'),
    widgets.HBox([progress_bar, progress_label])
])

display(header, widgets.HBox([out_image, right_panel]))
show_current()

In [ ]:
merged = {}

for filename, data in labels.items():
    if filename in final_labels:
        merged[filename] = final_labels[filename]
    else:
        merged[filename] = {**data, 'reviewer': 'llm'}

merged_path = '/content/drive/MyDrive/CDT_Dataset/labels/ccss_labels_final.json'
with open(merged_path, 'w') as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

import pandas as pd

rows = []
for filename, data in merged.items():
    scores = data.get('scores', {})
    rows.append({
        'filename':             filename,
        'domain_A':             scores.get('digit_count', data.get('A')),
        'domain_B':             scores.get('worst_quadrant_error', data.get('B')),
        'domain_C':             scores.get('spatial_arrangement', data.get('C')),
        'domain_D':             scores.get('hands_present', data.get('D')),
        'domain_E':             scores.get('hands_placement', data.get('E')),
        'total_score':          data.get('total_score', data.get('total')),
        'dementia_risk':        data.get('risk_flag', data.get('dementia_risk')),
        'confidence':           data.get('confidence'),
        'reviewer':             data.get('reviewer', 'llm'),
        'rotation_used':        data.get('rotation_used', 0),
    })

df = pd.DataFrame(rows)
csv_path = '/content/drive/MyDrive/CDT_Dataset/labels/ccss_labels_final.csv'
df.to_csv(csv_path, index=False, encoding='utf-8-sig')

print(f'Final dataset: {len(df)} รูป')
print(f'\nReviewer breakdown:')
print(df['reviewer'].value_counts())
print(f'\nDementia risk: {df["dementia_risk"].sum()} รูป ({df["dementia_risk"].mean()*100:.1f}%)')
print(f'ปกติ: {(~df["dementia_risk"]).sum()} รูป')
print(f'คะแนนเฉลี่ย: {df["total_score"].mean():.2f}')
print(f'\nบันทึกแล้วที่ {csv_path}')
display(df.head())

merge labels...
Final dataset: 6602 รูป

Reviewer breakdown:
reviewer
llm               4963
human             1108
human_approved     531
Name: count, dtype: int64

Dementia risk: 980 รูป (14.8%)
ปกติ: 5622 รูป
คะแนนเฉลี่ย: 8.28

บันทึกแล้วที่ /content/drive/MyDrive/CDT_Dataset/labels/ccss_labels_final.csv


,filename,domain_A,domain_B,domain_C,domain_D,domain_E,total_score,dementia_risk,confidence,reviewer,rotation_used
0,10000008.tif,2,1,2,2,2,9,False,0.92,llm,0
1,10000021.tif,2,1,2,2,2,9,False,0.92,llm,0
2,10000036.tif,2,1,2,2,2,9,False,0.92,llm,0
3,10000041.tif,2,1,2,0,0,5,True,0.85,human,0
4,10000043.tif,2,1,2,2,2,9,False,0.92,human_approved,0
